# Uso de GPU

In [22]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

I0000 00:00:1780078757.354344 1242472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 0 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
GPU 1 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}


# Análise demográfica

In [23]:
import pandas as pd

df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 525
Linhas totais: 1575
Checagem linhas/3: 525.0

Por GROUP:
GROUP
pMCI    128
sMCI    397
Name: count, dtype: int64

Por SEX:
SEX
F    222
M    303
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
pMCI    54   74  128
sMCI   168  229  397
All    222  303  525


In [24]:
df_ext.groupby("GROUP")["ID_PT"].nunique()

GROUP
pMCI    128
sMCI    397
Name: ID_PT, dtype: int64

In [25]:
dist_pacientes = (
    df_ext.drop_duplicates(subset=["ID_PT", "GROUP"])
    .groupby(["GROUP", "SEX"])["ID_PT"]
    .nunique()
    .unstack(fill_value=0)
)

dist_pacientes

SEX,F,M
GROUP,,
pMCI,54,74
sMCI,168,229


In [26]:
pts = df_ext.groupby(["GROUP", "ID_PT"], as_index=False)["AGE"].min()

for g in ["sMCI", "pMCI"]:
    a = pts.loc[pts["GROUP"] == g, "AGE"]
    print(
        f"{g}: n={a.count()}, "
        f"min={a.min():.1f}, max={a.max():.1f}, "
        f"media={a.mean():.2f}, desvio={a.std():.2f}"
    )

sMCI: n=397, min=55.0, max=91.0, media=73.52, desvio=7.48
pMCI: n=128, min=57.0, max=89.0, media=75.04, desvio=7.01


In [27]:
adnimerge = pd.read_csv("csvs/adnimerged.csv")
df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

cols = ["ID_IMG", "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE"]

adni_sub = adnimerge.loc[:, cols].copy()
adni_sub["ID_IMG"] = adni_sub["ID_IMG"].astype(str).str.strip()
adni_sub = adni_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

df_ext["ID_IMG"] = df_ext["ID_IMG"].astype(str).str.strip()
df_ext = df_ext.merge(adni_sub, on="ID_IMG", how="left", validate="many_to_one")

In [28]:
pts = (
    df_ext.sort_values(["GROUP", "ID_PT", "MRI_DATE"])
    .drop_duplicates(subset=["GROUP", "ID_PT"], keep="first")
)

pts.groupby("GROUP")[["MMSE_SCORE", "FAQ_SCORE", "CDR_GLOBAL", "ADAS_SCORE"]].agg(["mean", "std", "count"])

MMSE_SCORE                 FAQ_SCORE                 CDR_GLOBAL  \
            mean       std count      mean       std count       mean   
GROUP                                                                   
pMCI   26.039062  2.249768   128  6.453125  4.619131   128   0.507812   
sMCI   27.758186  1.822132   397  2.503778  3.573628   397   0.497481   

                      ADAS_SCORE                  
            std count       mean       std count  
GROUP                                             
pMCI   0.062253   128  14.382891  5.055354   128  
sMCI   0.035444   397   9.491788  3.917522   397

# Testes TODO

- estudar o treinamento dos modelos, se é por épocas ou tudo junto
- verificar como detectar overfitting para o modelo utilizado
- documentar o pipeline
- testar outros modelos, xgboost, random forest
- teste com todos os atributos
- filtro variancia + correlação
- filtro atributos monotonicos
- atributos radiomicos + deslocamento
- estudar wandb


In [20]:
# Nested CV 5×5 — comparação de modelos (pipeline unificado)
# Ordem fiel ao notebook: corr → var → kbest → scaler → classificador

import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message=".*greater than n_features.*",
    category=UserWarning,
    module="sklearn.feature_selection._univariate_selection",
)
# ==========================================
# CONFIGURAÇÕES
# ==========================================
SEED = 42
K_OUT, K_IN = 5, 5
PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/ablation_all_rad_disp_legacy_disp_article_temporal_hippocampus.csv"
TEST_NAME = "nested_cv_multimodel"

METRIC_COLS = [
    "accuracy","auc", "auc_pr", "bal_acc", "mcc",
    "sens_pMCI", "spec_sMCI", "prec_pMCI", "f1_pMCI",
    "recall_pMCI", "precision_pMCI",
]

# ==========================================
# FILTRO DE CORRELAÇÃO (igual ao notebook)
# ==========================================
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.to_keep_ = None

    def fit(self, X, y=None):
        xf = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if xf.shape[0] < 2:
            self.to_keep_ = np.arange(X.shape[1], dtype=int)
            return self

        c = np.corrcoef(xf.T)
        keep = []
        for j in range(c.shape[0]):
            if all(
                not (np.isfinite(c[j, k]) and abs(c[j, k]) > self.threshold)
                for k in keep
            ):
                keep.append(j)
        self.to_keep_ = np.asarray(keep, dtype=int)
        return self

    def transform(self, X):
        return X[:, self.to_keep_]


# ==========================================
# MÉTRICAS
# ==========================================
def fold_metrics(y_true, scores, pred) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "auc": float(roc_auc_score(y_true, scores)),
        "auc_pr": float(average_precision_score(y_true, scores)),
        "bal_acc": float(balanced_accuracy_score(y_true, pred)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "sens_pMCI": float(tp / (tp + fn)) if (tp + fn) else float("nan"),
        "spec_sMCI": float(tn / (tn + fp)) if (tn + fp) else float("nan"),
        "prec_pMCI": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        "f1_pMCI": float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
        "recall_pMCI": float(recall_score(y_true, pred, pos_label=1, zero_division=0)),
        "precision_pMCI": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
    }


def tune_youden_threshold(y_true, scores) -> float:
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y_true)) < 2:
        return 0.0
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    return float(thr[int(np.argmax(j))])


def get_scores(model, X):
    clf = model.named_steps["clf"]
    if hasattr(clf, "decision_function"):
        return model.decision_function(X)
    return model.predict_proba(X)[:, 1]


# def count_features_after_preprocess(model, X):
#     preprocess = Pipeline(model.steps[:-1])
#     return int(preprocess.transform(X).shape[1])
def count_features_after_preprocess(model, X):
    Xt = X
    for name, step in model.steps[:-1]:  # todos exceto o classificador
        if hasattr(step, "transform"):
            Xt = step.transform(Xt)
        # RandomUnderSampler e similares: ignorar (não mudam n_features)
    return int(Xt.shape[1])

def simplify_best_params(best_params: dict) -> dict:
    out = {}
    for k, v in best_params.items():
        if k == "clf":
            out["model"] = v.__class__.__name__
        elif k == "kbest" and v == "passthrough":
            out["kbest"] = "off"
        elif k == "kbest":
            out["kbest"] = "on"
        else:
            out[k.replace("clf__", "").replace("kbest__", "k=")] = v
    return out


# ==========================================
# NESTED CV — COMPARAÇÃO DE MODELOS
# ==========================================
def nested_cv_multimodel(X, y, test_name: str = TEST_NAME):
    results = []

    # base_pipeline = Pipeline([
    #     ("corr_filter", CorrelationFilter(threshold=0.95)),
    #     ("var_filter", VarianceThreshold(threshold=0.01)),
    #     ("kbest", "passthrough"),
    #     ("scaler", StandardScaler()),
    #     ("clf", LinearSVC(random_state=SEED)),  # placeholder
    # ])
    base_pipeline = ImbPipeline([
        ("corr_filter", CorrelationFilter(threshold=0.95)),
        ("var_filter", VarianceThreshold(threshold=0.01)),
        # --- DOWNSAMPLING ENTRA AQUI ---
        ("sampler", RandomUnderSampler(random_state=SEED)), 
        ("kbest", "passthrough"),
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(random_state=SEED)),  # placeholder
    ])
    outer_cv = StratifiedKFold(K_OUT, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        pos_weight = float(np.sum(y_train == 0) / np.sum(y_train == 1))
        svm = LinearSVC(
            penalty="l1",
            dual=False,
            class_weight="balanced",
            max_iter=200_000,
            random_state=SEED,
        )

        param_grid = [
            # 1) LinearSVC — réplica do Teste 1 (sem SelectKBest)
            {
                "kbest": ["passthrough"],
                "clf": [svm],
                "clf__C": [1e-3, 0.01, 0.1, 1.0, 10.0],
            },
            # # 2) LinearSVC + SelectKBest
            # {
            #     "kbest": [SelectKBest(score_func=f_classif)],
            #     "kbest__k": [30, 50, 100, 200],
            #     "clf": [svm],
            #     "clf__C": [1e-3, 0.01, 0.1, 1.0, 10.0],
            # },
            # 3) Random Forest sem SelectKBest
            # {
            #     "kbest": ["passthrough"],
            #     "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
            #     "clf__n_estimators": [100, 200],
            #     "clf__max_depth": [None, 5, 10],
            # },
            # # 4) Random Forest com SelectKBest
            # {
            #     "kbest": [SelectKBest(score_func=f_classif)],
            #     "kbest__k": [50, 100, 200],
            #     "clf": [RandomForestClassifier(class_weight="balanced", random_state=SEED)],
            #     "clf__n_estimators": [100, 200],
            #     "clf__max_depth": [None, 5, 10],
            # },
            # 5) XGBoost sem SelectKBest
            # {
            #     "kbest": ["passthrough"],
            #     "clf": [XGBClassifier(
            #         # scale_pos_weight=pos_weight,
            #         eval_metric="logloss",
            #         random_state=SEED,
            #     )],
            #     "clf__n_estimators": [100, 200],
            #     "clf__learning_rate": [0.01, 0.1],
            #     "clf__max_depth": [3, 5],
            # },
            # # 6) XGBoost com SelectKBest
            # {
            #     "kbest": [SelectKBest(score_func=f_classif)],
            #     "kbest__k": [50, 100, 200],
            #     "clf": [XGBClassifier(
            #         scale_pos_weight=pos_weight,
            #         eval_metric="logloss",
            #         random_state=SEED,
            #     )],
            #     "clf__n_estimators": [100, 200],
            #     "clf__learning_rate": [0.01, 0.1],
            #     "clf__max_depth": [3, 5],
            # },
        ]

        inner_cv = StratifiedKFold(K_IN, shuffle=True, random_state=SEED + fold)

        grid_search = GridSearchCV(
            estimator=base_pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            refit=True,
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        model_name = best_model.named_steps["clf"].__class__.__name__

        # Limiar Youden com scores out-of-fold (sem vazar teste)
        if hasattr(best_model.named_steps["clf"], "decision_function"):
            train_scores = cross_val_predict(
                best_model, X_train, y_train, cv=inner_cv, method="decision_function"
            )
        else:
            train_scores = cross_val_predict(
                best_model, X_train, y_train, cv=inner_cv, method="predict_proba"
            )[:, 1]

        threshold = tune_youden_threshold(y_train, train_scores)

        test_scores = get_scores(best_model, X_test)
        test_preds = (test_scores >= threshold).astype(int)

        row = {
            "test": test_name,
            "fold": fold,
            "best_model": model_name,
            "best_inner_auc": float(grid_search.best_score_),
            "best_params": simplify_best_params(grid_search.best_params_),
            "threshold": threshold,
            "n_features": count_features_after_preprocess(best_model, X_train),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_test_pMCI": int(y_test.sum()),
            **fold_metrics(y_test, test_scores, test_preds),
        }
        results.append(row)

        print(
            f"Fold {fold} | vencedor: {model_name} | "
            f"AUC interno: {row['best_inner_auc']:.3f} | "
            f"AUC teste: {row['auc']:.3f}"
        )

    return pd.DataFrame(results)


def print_summary(df: pd.DataFrame) -> None:
    print(f"\n=== {df['test'].iloc[0]} ===")
    display(df)

    print("\nResumo geral (média ± desvio entre folds):")
    for col in METRIC_COLS:
        print(f"  {col:14s}: {df[col].mean():.3f} ± {df[col].std():.3f}")

    print("\nModelo vencedor por fold:")
    display(df.groupby("best_model").size().rename("n_folds").to_frame())

    print("\nMédia de AUC no teste por modelo vencedor:")
    display(
        df.groupby("best_model")["auc"]
        .agg(["count", "mean", "std"])
        .rename(columns={"count": "n_folds", "mean": "auc_mean", "std": "auc_std"})
    )

In [21]:
# Carregar dados e rodar
meta = ["ID_PT", "GROUP", "SEX"]
df = pd.read_csv(PATH)

X = df.drop(columns=meta).to_numpy(float)
y = df["GROUP"].to_numpy(int)

print(f"{TEST_NAME} | {len(y)} pacientes | {X.shape[1]} features originais")
print(f"Classes: sMCI={(y == 0).sum()} | pMCI={(y == 1).sum()}\n")

df_results = nested_cv_multimodel(X, y, test_name=TEST_NAME)
print_summary(df_results)

nested_cv_multimodel | 525 pacientes | 1368 features originais
Classes: sMCI=397 | pMCI=128

Fold 1 | vencedor: LinearSVC | AUC interno: 0.760 | AUC teste: 0.730
Fold 2 | vencedor: LinearSVC | AUC interno: 0.737 | AUC teste: 0.667
Fold 3 | vencedor: LinearSVC | AUC interno: 0.713 | AUC teste: 0.852
Fold 4 | vencedor: LinearSVC | AUC interno: 0.709 | AUC teste: 0.746
Fold 5 | vencedor: LinearSVC | AUC interno: 0.778 | AUC teste: 0.667

=== nested_cv_multimodel ===


,test,fold,best_model,best_inner_auc,best_params,threshold,n_features,n_train,n_test,n_test_pMCI,...,auc,auc_pr,bal_acc,mcc,sens_pMCI,spec_sMCI,prec_pMCI,f1_pMCI,recall_pMCI,precision_pMCI
0,nested_cv_multimodel,1,LinearSVC,0.760241,"{'model': 'LinearSVC', 'C': 0.1, 'kbest': 'off'}",-0.000245,354,420,105,26,...,0.730282,0.444906,0.700584,0.354932,0.692308,0.708861,0.439024,0.537313,0.692308,0.439024
1,nested_cv_multimodel,2,LinearSVC,0.736539,"{'model': 'LinearSVC', 'C': 0.01, 'kbest': 'off'}",-0.004614,354,420,105,26,...,0.666504,0.470612,0.662366,0.285033,0.653846,0.670886,0.395349,0.492754,0.653846,0.395349
2,nested_cv_multimodel,3,LinearSVC,0.713256,"{'model': 'LinearSVC', 'C': 0.1, 'kbest': 'off'}",0.163345,349,420,105,26,...,0.851996,0.619758,0.750974,0.484566,0.653846,0.848101,0.586207,0.618182,0.653846,0.586207
3,nested_cv_multimodel,4,LinearSVC,0.709170,"{'model': 'LinearSVC', 'C': 0.01, 'kbest': 'off'}",-0.005324,355,420,105,25,...,0.746000,0.540461,0.678750,0.306216,0.720000,0.637500,0.382979,0.500000,0.720000,0.382979
4,nested_cv_multimodel,5,LinearSVC,0.778437,"{'model': 'LinearSVC', 'C': 0.1, 'kbest': 'off'}",-0.204757,343,420,105,25,...,0.667000,0.435933,0.636250,0.233410,0.760000,0.512500,0.327586,0.457831,0.760000,0.327586



Resumo geral (média ± desvio entre folds):
  accuracy      : 0.680 ± 0.083
  auc           : 0.732 ± 0.076
  auc_pr        : 0.502 ± 0.077
  bal_acc       : 0.686 ± 0.043
  mcc           : 0.333 ± 0.095
  sens_pMCI     : 0.696 ± 0.045
  spec_sMCI     : 0.676 ± 0.121
  prec_pMCI     : 0.426 ± 0.098
  f1_pMCI       : 0.521 ± 0.061
  recall_pMCI   : 0.696 ± 0.045
  precision_pMCI: 0.426 ± 0.098

Modelo vencedor por fold:


,n_folds
best_model,
LinearSVC,5



Média de AUC no teste por modelo vencedor:


,n_folds,auc_mean,auc_std
best_model,,,
LinearSVC,5,0.732357,0.076013
